# Project Reproducibility and Paper-Alignment Checks

This executed notebook validates the source scale, six-stage funnel, chronological split, model metrics, feature contract, repository structure, and visible notebook outputs.

## 1. Load the external session-level CSV

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
data_candidates = [
    ROOT / "data" / "session_level_ecommerce.csv.gz",
    ROOT / "data" / "session_level_ecommerce.csv",
    ROOT / "session_level_ecommerce.csv.gz",
    ROOT / "session_level_ecommerce.csv",
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("External session-level CSV not found.")
df = pd.read_csv(DATA_PATH)
print(f"Validating {len(df):,} sessions from {DATA_PATH}")

Validating 310,014 sessions from data/session_level_ecommerce.csv.gz


## 2. Validate core paper metrics and the complete funnel

In [2]:
expected_funnel = {
    "Visit": 310_014,
    "Product View": 65_393,
    "Add to Cart": 15_187,
    "Begin Checkout": 9_086,
    "Payment Info": 5_875,
    "Purchase": 4_247,
}
actual_funnel = {
    "Visit": len(df),
    "Product View": int(df["viewed_product"].sum()),
    "Add to Cart": int(df["added_to_cart"].sum()),
    "Begin Checkout": int(df["began_checkout"].sum()),
    "Payment Info": int(df["added_payment_info"].sum()),
    "Purchase": int(df["converted"].sum()),
}
assert int(df["total_events"].sum()) == 3_624_334
assert int(df["user_pseudo_id"].nunique()) == 235_981
assert len(df) == 310_014
assert df["session_id"].is_unique
assert int(df["converted"].sum()) == 4_247
assert np.isclose(df["converted"].mean(), 0.0136993813, atol=1e-8)
assert np.isclose(df["revenue"].sum(), 315_948, atol=1)
assert actual_funnel == expected_funnel
core_checks = pd.DataFrame({
    "check": [
        "Events analyzed = 3,624,334", "Unique users = 235,981",
        "Sessions = 310,014", "Session IDs are unique",
        "Purchases = 4,247", "Conversion rate = 1.3699%",
        "Revenue = $315,948", "Six-stage funnel matches the paper",
    ],
    "passed": [True] * 8,
})
display(core_checks)
print("Core data and funnel metrics validated successfully.")

check,passed
"Events analyzed = 3,624,334",True
"Unique users = 235,981",True
"Sessions = 310,014",True
Session IDs are unique,True
"Purchases = 4,247",True
Conversion rate = 1.3699%,True
"Revenue = $315,948",True
Six-stage funnel matches the paper,True


Core data and funnel metrics validated successfully.


## 3. Validate chronological model results

In [3]:
metrics = json.loads(
    (ROOT / "artifacts" / "model_metrics.json").read_text(encoding="utf-8")
)
expected_model_results = {
    "hashed_logistic_regression": {
        "pr_auc": 0.0171, "roc_auc": 0.5868,
        "log_loss": 0.0609, "brier_score": 0.01105,
        "lift_top_10pct": 2.01,
    },
    "categorical_naive_bayes": {
        "pr_auc": 0.0168, "roc_auc": 0.5947,
        "log_loss": 0.0631, "brier_score": 0.01138,
        "lift_top_10pct": 2.03,
    },
}
for model_name, expected in expected_model_results.items():
    actual = metrics["models"][model_name]
    for metric, expected_value in expected.items():
        tolerance = 0.02 if metric == "lift_top_10pct" else 0.0001
        assert np.isclose(actual[metric], expected_value, atol=tolerance)
assert metrics["data"]["train_rows"] == 228_487
assert metrics["data"]["train_purchases"] == 3_335
assert metrics["data"]["test_rows"] == 81_527
assert metrics["data"]["test_purchases"] == 912
assert metrics["selected_by_pr_auc"] == "hashed_logistic_regression"
assert len(metrics["feature_contract"]["safe_features"]) == 7
assert metrics["feature_contract"]["status"] == "passed"
model_checks = pd.DataFrame({
    "check": [
        "Training sessions / purchases = 228,487 / 3,335",
        "Test sessions / purchases = 81,527 / 912",
        "Hashed logistic regression metrics match",
        "Categorical Naive Bayes metrics match",
        "Selected model is hashed logistic regression",
        "Seven-feature leakage contract passed",
    ],
    "passed": [True] * 6,
})
display(model_checks)
print("Model metrics validated successfully.")

check,passed
"Training sessions / purchases = 228,487 / 3,335",True
"Test sessions / purchases = 81,527 / 912",True
Hashed logistic regression metrics match,True
Categorical Naive Bayes metrics match,True
Selected model is hashed logistic regression,True
Seven-feature leakage contract passed,True


Model metrics validated successfully.


## 4. Validate repository alignment and visible outputs

In [4]:
integrated_path = ROOT / "Ecommerce_Conversion_Analytics_Integrated.ipynb"
integrated = json.loads(integrated_path.read_text(encoding="utf-8"))
code_cells = [cell for cell in integrated["cells"] if cell["cell_type"] == "code"]
checks = {
    "Canonical integrated notebook exists": integrated_path.exists(),
    "Every integrated-notebook code cell has output": all(cell["outputs"] for cell in code_cells),
    "Complete session-reconstruction SQL exists": (ROOT / "sql" / "01_build_session_table.sql").exists(),
    "Seven-feature session-start SQL exists": (ROOT / "sql" / "02_build_session_start_features.sql").exists(),
    "Five-minute BQML extension is isolated": (ROOT / "experiments" / "02_train_five_minute_bqml_models.sql").exists(),
    "No Python script is required for notebook execution": not any(ROOT.rglob("*.py")),
}
repository_checks = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
display(repository_checks)
assert all(checks.values())
print("Repository and visible-output checks passed.")

check,passed
Canonical integrated notebook exists,True
Every integrated-notebook code cell has output,True
Complete session-reconstruction SQL exists,True
Seven-feature session-start SQL exists,True
Five-minute BQML extension is isolated,True
No Python script is required for notebook execution,True


Repository and visible-output checks passed.
